# [267] Virtual team 404

**Full names:**
- Marius Aleksander Brun (stud ID: 109518)
- Thomas Mogstad Barth (stud ID: 172010)
- Amund Dahlmo Berge (stud ID: 109546)

**Kaggle team name:** [267] Virtual team 404

**Model:** XGBoost on one row per (case, hour) with 133 features (calendar, price, inflow, volumes, water values, price-minus-water-value margins, minimum-flow and minimum-volume constraints). Settings (maximum depth 6, minimum child weight 100, L2 regularisation 20) and rounds (368) were chosen by a grid search on 2021; the final model is trained on all cases. Run from the folder that contains `data/` and `prediction_mapping.csv`.

In [1]:
import shutil
import numpy as np
import pandas as pd
import xgboost as xgb
from sklearn.metrics import roc_auc_score


In [2]:
H = 168

def read_table(path, hourly=False):
    table = pd.read_csv(path)
    table['date'] = pd.to_datetime(table['date'], utc=True).dt.tz_localize(None)
    table = table.set_index('date')
    return table.resample('h').mean() if hourly else table

train = pd.read_csv('data/kernel/Unit_commitment_decisions.csv')
train['starttime'] = pd.to_datetime(train['starttime'], utc=True).dt.tz_localize(None)
target_columns = pd.read_csv('sample_submission.csv', nrows=1).columns[1:]
test = pd.read_csv('prediction_mapping.csv')
test['starttime'] = pd.to_datetime(test['starttime'], utc=True).dt.tz_localize(None)

price = read_table('data/kernel/Historical_day_ahead_price_2015_2025.csv', hourly=True)['Day-ahead price (EUR/MWh)']
inflow = read_table('data/kernel/Historical_inflow_1958_2025.csv')
inflow = inflow[inflow.index >= '2014-12-01'].resample('h').mean()
volume = read_table('data/kernel/Historical_volume_2015_2024.csv')
water_value = read_table('data/kernel/Synthetic_water_value_2015_2024.csv')
min_flow = read_table('data/extended/Constraint_min_flow.csv', hourly=True)
min_volume = read_table('data/extended/Constraint_min_volume.csv', hourly=True)

In [3]:
# reservoir directly upstream of each plant (from Tokke_Vinje_topology.yaml)
upstream = {
    'Hogga': 'Bandak', 'Tokke': 'Vinjevatn', 'Lio': 'Byrtevatn', 'Byrte': 'Botnedalsvatn',
    'Vinje': 'Vaamarvatn', 'Haukeli': 'Vatjern', 'Songa': 'Songavatn', 'Kjela': 'Foersvatn',
    'Vesle Kjela': 'Kjelavatn',
}

def weekly_window(table, starts):
    idx = table.index.get_indexer(starts)
    assert (idx >= 0).all()
    return table.to_numpy(dtype=np.float32)[idx[:, None] + np.arange(H)[None, :]]

def make_features(starts):
    starts = pd.DatetimeIndex(starts)
    n = len(starts)
    rep = lambda a: np.repeat(a, H)
    P = weekly_window(price.to_frame(), starts)[:, :, 0]
    I = weekly_window(inflow, starts)
    V = volume.reindex(starts).to_numpy(dtype=np.float32)
    W = water_value.reindex(starts).to_numpy(dtype=np.float32)
    W_end = water_value.reindex(starts + pd.Timedelta(days=7)).to_numpy(dtype=np.float32)
    hours = pd.DatetimeIndex((starts.values[:, None] + np.arange(H)[None, :] * np.timedelta64(1, 'h')).ravel())

    f = pd.DataFrame({
        't': np.tile(np.arange(H), n), 'hour': hours.hour, 'dow': hours.dayofweek,
        'month': hours.month, 'doy': hours.dayofyear,
        'price': P.ravel(), 'price_week_mean': rep(P.mean(1)), 'price_week_std': rep(P.std(1)),
        'price_week_min': rep(P.min(1)), 'price_week_max': rep(P.max(1)),
    })
    f['price_rel'] = f['price'] - f['price_week_mean']
    f['price_rank'] = (P.argsort(1).argsort(1) / (H - 1)).ravel()
    f['price_day_mean'] = np.repeat(P.reshape(n, 7, 24).mean(2), 24, axis=1).ravel()
    f['price_prev'] = np.concatenate([P[:, :1], P[:, :-1]], axis=1).ravel()
    f['price_next'] = np.concatenate([P[:, 1:], P[:, -1:]], axis=1).ravel()
    for j, name in enumerate(inflow.columns):
        f[f'inflow_{name}'] = I[:, :, j].ravel()
    f['inflow_total'] = I.sum(2).ravel()
    for j, name in enumerate(volume.columns):
        f[f'vol_{name}'] = rep(V[:, j])
    for j, name in enumerate(water_value.columns):
        f[f'wv_{name}'] = rep(W[:, j])
    for j, name in enumerate(water_value.columns):
        f[f'wv_end_{name}'] = rep(W_end[:, j])

    # margins: price minus the water value of the upstream reservoir
    margins = {}
    for plant, res in upstream.items():
        margins[f'margin_{plant}'] = f['price'] - f[f'wv_{res}']
        margins[f'margin_end_{plant}'] = f['price'] - f[f'wv_end_{res}']
        margins[f'margin_end_wk_{plant}'] = f['price_week_mean'] - f[f'wv_end_{res}']

    # constraints: minimum flow per river, minimum volume and slack above it per reservoir
    MF = weekly_window(min_flow, starts)
    MV = weekly_window(min_volume, starts)
    V0 = volume.reindex(starts)
    extra = {}
    for j, river in enumerate(min_flow.columns):
        extra[f'minflow_{river}'] = MF[:, :, j].ravel()
        extra[f'minflow_{river}_wkmax'] = rep(MF[:, :, j].max(1))
    for j, name in enumerate(min_volume.columns):
        v0 = V0[name].to_numpy(dtype=np.float32)
        extra[f'minvol_{name}'] = MV[:, :, j].ravel()
        extra[f'volgap_{name}'] = rep(v0) - MV[:, :, j].ravel()
        extra[f'volgap_{name}_wkmin'] = rep(v0 - MV[:, :, j].max(1))
    return pd.concat([f, pd.DataFrame(margins), pd.DataFrame(extra)], axis=1)

In [4]:
X = make_features(train['starttime'])
X_test = make_features(test['starttime'])
y = train[target_columns].to_numpy(dtype=np.int8).reshape(-1, 14, H).transpose(0, 2, 1).reshape(-1, 14)

# GPU if the machine has one (this reproduces the delivered file exactly), otherwise the CPU (predictions then differ slightly)
device = 'cuda' if shutil.which('nvidia-smi') else 'cpu'

model = xgb.XGBClassifier(
    n_estimators=368, learning_rate=0.05, max_depth=6, subsample=0.8, colsample_bytree=0.8,
    min_child_weight=100, reg_lambda=20, tree_method='hist', device=device, random_state=42,
)
model.fit(X, y)

wide = model.predict_proba(X_test).reshape(-1, H, 14).transpose(0, 2, 1).reshape(len(test), -1)
submission = pd.DataFrame(wide, columns=target_columns)
submission.insert(0, 'Run No', test['Run No'].to_numpy())
submission.to_csv('2026-11-05_xgb_gridsearch_cv0.9810.csv', index=False)

# summary of the test predictions (the test cases have no labels, so there is no test score here)
units = list(dict.fromkeys(c.rsplit('_t', 1)[0].replace('result_committed_', '') for c in target_columns))
print(f'{len(submission)} test cases x {len(target_columns)} predictions written, device: {device}')
print(f'predicted ON probability: min {wide.min():.3f}, max {wide.max():.3f}, mean {wide.mean():.3f}')
summary = pd.DataFrame({
    'ON rate in training data': y.mean(axis=0),
    'mean predicted ON, test': wide.reshape(len(test), 14, H).mean(axis=(0, 2)),
}, index=units).round(3)
summary


C:\Users\thoma\AppData\Local\Programs\Python\Python312\Lib\site-packages\xgboost\core.py:569: UserWarning: [14:03:05] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\common\error_msg.cc:62: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.

  return func(**kwargs)


725 test cases x 2352 predictions written, device: cuda
predicted ON probability: min 0.000, max 1.000, mean 0.468


,ON rate in training data,"mean predicted ON, test"
Hogga_G1,0.595,0.618
Tokke_G1,0.671,0.589
Tokke_G2,0.478,0.408
Tokke_G3,0.564,0.494
Tokke_G4,0.498,0.428
Lio_G1,0.747,0.748
Byrte_G1,0.542,0.587
Vinje_G1,0.485,0.354
Vinje_G2,0.420,0.296
Vinje_G3,0.259,0.191


In [5]:
# validation score on 2022: same settings and 368 rounds (chosen on 2021 by the grid search), trained only on cases before 2022
starts = train['starttime']
fit_rows = np.repeat((starts < pd.Timestamp('2022-01-01') - pd.Timedelta(days=6)).to_numpy(), H)   # one-week gap
val_rows = np.repeat((starts.dt.year == 2022).to_numpy(), H)

val_model = xgb.XGBClassifier(**model.get_params()).fit(X[fit_rows], y[fit_rows])
val_pred = val_model.predict_proba(X[val_rows])

print(f'2022 validation, micro ROC-AUC: {roc_auc_score(y[val_rows], val_pred, average="micro"):.4f}')
pd.Series(
    [roc_auc_score(y[val_rows][:, j], val_pred[:, j]) for j in range(14)], index=units, name='2022 ROC-AUC'
).round(3).sort_values().to_frame()

2022 validation, micro ROC-AUC: 0.9810


,2022 ROC-AUC
Lio_G1,0.906
Haukeli_G1,0.912
Vinje_G3,0.967
Hogga_G1,0.974
Kjela_G1,0.980
Vinje_G2,0.980
Vesle Kjela_G1,0.981
Vinje_G1,0.984
Tokke_G2,0.987
Tokke_G4,0.987
